# 🧠 Lab 5.0 — ML Performance Evaluation
### Classification and Regression | Artificial Intelligence 5.0

---

This notebook covers:
- **Regression Problem** → Predicting *Sleep Duration* (continuous value)
- **Classification Problem** → Predicting *Sleep Disorder* (categorical outcome)

Dataset: **Sleep Health and Lifestyle Dataset**


## 📦 Install Dependencies

In [ ]:
# Run this cell if any library is missing
# !pip install pandas scikit-learn matplotlib seaborn


## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")


## 2️⃣ Load the Dataset
> **Update the path below** to point to your CSV file.


In [ ]:
\
# ── UPDATE THIS PATH ──────────────────────────────────────────────────────
CSV_PATH = r"C:\Users\markl\ARDUINO-TEAM_SINCO\p1classification_model\regress\sleep\Sleep_health_and_lifestyle_dataset.csv"
# ─────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## 3️⃣ Exploratory Data Analysis

In [ ]:
# Data types and missing values
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Missing Values ===")
print(df.isnull().sum())


In [ ]:
# Basic statistics
df.describe()


## 4️⃣ Data Preparation
### Handle Missing Values & Feature Engineering


In [ ]:
# Make a clean copy
data = df.copy()

# Split Blood Pressure into systolic and diastolic
data[["BP_Systolic", "BP_Diastolic"]] = (
    data["Blood Pressure"].str.split("/", expand=True).astype(int)
)

# Fill missing Sleep Disorder with "None" (no disorder)
data["Sleep Disorder"] = data["Sleep Disorder"].fillna("None")

# Drop columns not needed as features
data = data.drop(columns=["Person ID", "Blood Pressure"])

print("✅ Preprocessing done!")
print(f"Shape after preprocessing: {data.shape}")
data.head()


### Encode Categorical Features

In [ ]:
# One-hot encode for regression (drop_first avoids multicollinearity)
data_encoded = pd.get_dummies(
    data,
    columns=["Gender", "Occupation", "BMI Category", "Sleep Disorder"],
    drop_first=True
)

print(f"Shape after encoding: {data_encoded.shape}")
data_encoded.head()


---
# 📈 PART A — REGRESSION PROBLEM
## Predicting Sleep Duration (Continuous Value)


## 5️⃣ Feature & Target Selection

In [ ]:
X_reg = data_encoded.drop(columns=["Sleep Duration"])
y_reg = data_encoded["Sleep Duration"]

print(f"Features: {X_reg.shape[1]}")
print(f"Target: Sleep Duration — sample values: {y_reg.values[:5]}")


## 6️⃣ Train / Test Split

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train_r.shape[0]}")
print(f"Testing  samples : {X_test_r.shape[0]}")


## 7️⃣ Scale Features

In [ ]:
scaler_r = StandardScaler()
X_train_r_sc = scaler_r.fit_transform(X_train_r)
X_test_r_sc  = scaler_r.transform(X_test_r)


## 8️⃣ Build & Train the Model — Linear Regression

In [ ]:
reg_model = LinearRegression()
reg_model.fit(X_train_r_sc, y_train_r)

print("✅ Linear Regression model trained!")


## 9️⃣ Evaluate the Regression Model

In [ ]:
y_pred_r = reg_model.predict(X_test_r_sc)

mae  = mean_absolute_error(y_test_r, y_pred_r)
mse  = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2   = r2_score(y_test_r, y_pred_r)

print("=== Regression Metrics ===")
print(f"MAE  : {mae:.4f} hours  → avg prediction error")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f} hours")
print(f"R²   : {r2:.4f}  → {r2*100:.1f}% of variance explained")


### Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
axes[0].scatter(y_test_r, y_pred_r, alpha=0.6, color="steelblue", edgecolors="white", linewidth=0.5)
axes[0].plot([y_test_r.min(), y_test_r.max()],
             [y_test_r.min(), y_test_r.max()], "r--", lw=2, label="Perfect prediction")
axes[0].set_xlabel("Actual Sleep Duration (hrs)")
axes[0].set_ylabel("Predicted Sleep Duration (hrs)")
axes[0].set_title("Actual vs Predicted — Sleep Duration")
axes[0].legend()

# Residuals
residuals = y_test_r - y_pred_r
axes[1].hist(residuals, bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual (hours)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()


### Top Feature Coefficients

In [ ]:
coef_df = (
    pd.DataFrame({"Feature": X_reg.columns, "Coefficient": reg_model.coef_})
    .assign(Abs=lambda d: d["Coefficient"].abs())
    .sort_values("Abs", ascending=False)
    .head(10)
)

colors = ["#2ECC71" if c > 0 else "#E74C3C" for c in coef_df["Coefficient"]]

plt.figure(figsize=(10, 5))
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient (standardized)")
plt.title("Top 10 Feature Coefficients — Linear Regression")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(coef_df[["Feature", "Coefficient"]].to_string(index=False))


---
# 🗂️ PART B — CLASSIFICATION PROBLEM
## Predicting Sleep Disorder (Categorical Outcome)


## 1️⃣0️⃣ Prepare for Classification

In [ ]:
# Encode target (Sleep Disorder) as numbers
le = LabelEncoder()
data["Sleep Disorder Encoded"] = le.fit_transform(data["Sleep Disorder"])

print("Classes:", list(le.classes_))
print(data["Sleep Disorder"].value_counts())


In [ ]:
# One-hot encode features (exclude Sleep Disorder from features)
data_clf = pd.get_dummies(
    data.drop(columns=["Sleep Disorder", "Sleep Disorder Encoded"]),
    columns=["Gender", "Occupation", "BMI Category"],
    drop_first=True
)

X_clf = data_clf
y_clf = data["Sleep Disorder Encoded"]

print(f"Features: {X_clf.shape[1]}")
print(f"Target classes: {le.classes_}")


## 1️⃣1️⃣ Train / Test Split

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"Training samples : {X_train_c.shape[0]}")
print(f"Testing  samples : {X_test_c.shape[0]}")


## 1️⃣2️⃣ Build & Train the Model — Random Forest Classifier

In [ ]:
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)

print("✅ Random Forest Classifier trained!")


## 1️⃣3️⃣ Evaluate the Classification Model

In [ ]:
y_pred_c = clf_model.predict(X_test_c)

acc  = accuracy_score(y_test_c, y_pred_c)
prec = precision_score(y_test_c, y_pred_c, average="weighted")
rec  = recall_score(y_test_c, y_pred_c, average="weighted")
f1   = f1_score(y_test_c, y_pred_c, average="weighted")

print("=== Classification Metrics ===")
print(f"Accuracy  : {acc:.4f}  → {acc*100:.1f}% correct predictions")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print("=== Classification Report ===")
print(classification_report(y_test_c, y_pred_c, target_names=le.classes_))


### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_c, y_pred_c)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Sleep Disorder Classification")
plt.tight_layout()
plt.show()


### Feature Importance

In [ ]:
feat_imp = (
    pd.DataFrame({"Feature": X_clf.columns, "Importance": clf_model.feature_importances_})
    .sort_values("Importance", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
plt.barh(feat_imp["Feature"], feat_imp["Importance"], color="#3498DB")
plt.xlabel("Importance Score")
plt.title("Top 10 Feature Importances — Random Forest")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(feat_imp.to_string(index=False))


---
## ✅ Summary

| | **Regression** | **Classification** |
|---|---|---|
| **Problem** | Predict Sleep Duration | Predict Sleep Disorder |
| **Output type** | Continuous (hours) | Categorical (None / Insomnia / Sleep Apnea) |
| **Algorithm** | Linear Regression | Random Forest |
| **Key metrics** | MAE, MSE, RMSE, R² | Accuracy, Precision, Recall, F1 |
| **Visualization** | Actual vs Predicted, Residuals | Confusion Matrix |

---
*Lab 5.0 — Artificial Intelligence | rgj*
